# 02. Preprocessing & Feature Embedding Extraction

Notebook ini melakukan pembentukan **Target Profit +3.0%**, ekstraksi **Chart Pattern & Feature Embeddings** (RSI Norm, MACD Diff, SMA Ratios, ATR Volatility, Multi-period Return Velocity), serta pembuatan dataset `X_train.csv`, `X_test.csv`, `y_train.csv`, dan `y_test.csv` tanpa One-Hot Encoding Ticker kaku.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

# Set project root path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import TICKERS, PROFIT_THRESHOLD
from src.features.technical_indicators import add_technical_indicators
from src.features.embedding import extract_chart_feature_embeddings

print(f"Total Tickers in Universe: {len(TICKERS)}")
print(f"Target Profit Threshold: {PROFIT_THRESHOLD * 100:.1f}%")

In [ ]:
# Fetch IHSG Benchmark Returns
try:
    ihsg = yf.download('^JKSE', start='2020-01-01', progress=False)
    ihsg_close = ihsg['Close'].iloc[:, 0] if isinstance(ihsg.columns, pd.MultiIndex) else ihsg['Close']
    ihsg_returns = pd.DataFrame({'IHSG_Return': ihsg_close.pct_change()}, index=ihsg.index)
    if ihsg_returns.index.tz is not None:
        ihsg_returns.index = ihsg_returns.index.tz_localize(None)
except Exception as e:
    print(f"Error fetching IHSG: {e}")
    ihsg_returns = pd.DataFrame()

print("IHSG Data Ready.")

In [ ]:
# Download Historical Price Data & Build Features
all_dfs = []
sample_tickers = TICKERS[:50]

for ticker in sample_tickers:
    try:
        df = yf.download(ticker, start='2020-01-01', progress=False)
        if df.empty or len(df) < 100:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel('Ticker') if 'Ticker' in df.columns.names else df.columns.get_level_values(0)
            
        df = add_technical_indicators(df)
        df['Return_1d'] = df['Close'].pct_change(1, fill_method=None)
        df['Return_2d'] = df['Close'].pct_change(2, fill_method=None)
        df['Return_3d'] = df['Close'].pct_change(3, fill_method=None)
        df['Return_5d'] = df['Close'].pct_change(5, fill_method=None)
        df['Day_of_Week'] = df.index.dayofweek
        
        if not ihsg_returns.empty:
            if df.index.tz is not None:
                df.index = df.index.tz_localize(None)
            df = df.join(ihsg_returns, how='left')
            df['IHSG_Return'] = df['IHSG_Return'].fillna(0)
        else:
            df['IHSG_Return'] = 0.0
            
        next_open = df['Open'].shift(-1)
        next_high = df['High'].shift(-1)
        df['Target'] = ((next_high - next_open) / next_open >= PROFIT_THRESHOLD).astype(int)
        
        df = df.iloc[:-1].copy()
        df['Ticker'] = ticker
        all_dfs.append(df)
    except Exception:
        continue

full_df = pd.concat(all_dfs)
full_df.dropna(subset=['Target', 'Close', 'Open', 'High', 'Low'], inplace=True)
print(f"Total Processed Dataset Rows: {len(full_df)}")

In [ ]:
# Extract Feature Embeddings
embed_df = extract_chart_feature_embeddings(full_df)
feature_cols = [
    'RSI_14', 'MACD_Diff', 'SMA_20', 'SMA_50', 'ATR_14',
    'Return_1d', 'Return_2d', 'Return_3d', 'Return_5d',
    'Embed_RSI_Norm', 'Embed_MACD_Diff', 'Embed_SMA20_Ratio',
    'Embed_SMA50_Ratio', 'Embed_Volatility_ATR', 'Embed_Return_1d',
    'Embed_Return_2d', 'Embed_Return_3d', 'Embed_Return_5d',
    'Embed_Log_Volume', 'Embed_IHSG_Return'
]

X = pd.concat([full_df, embed_df], axis=1)[feature_cols]
y = full_df['Target']

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# Split Train and Test Sets (80% Train, 20% Test Time-based)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / 'X_train.csv')
X_test.to_csv(processed_dir / 'X_test.csv')
y_train.to_csv(processed_dir / 'y_train.csv')
y_test.to_csv(processed_dir / 'y_test.csv')

print(f"Saved X_train: {X_train.shape}, X_test: {X_test.shape}")